### project 1 : google pay expenses sharing

In [124]:
!pip install numpy prettytable

In [125]:
import numpy as np
from prettytable import PrettyTable

In [126]:
friends = ["Madhan", "Gokul", "Loki", "Yusuf"] # 0, 1, 2, 3

# Rows: Who paid
# Columns: Who the expense was for
expense_matrix = np.zeros((len(friends), len(friends)))

In [127]:
expense_matrix

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [128]:
def add_expense(payer, beneficiaries, amount):
    payer_idx = friends.index(payer) #0
    
    #Calculate equal share
    share_per_person = amount / len(beneficiaries)
    
    #Update the expense matrix
    for beneficiary in beneficiaries:
        beneficiary_idx = friends.index(beneficiary) # 0, 1, 2, 3
        expense_matrix[payer_idx][beneficiary_idx] = expense_matrix[payer_idx][beneficiary_idx] + share_per_person

In [129]:
def calculate_settlements():
    # Total amount each person paid (Row Sum)                     axis 0 = column , axis 1 = rows
    total_paid = np.sum(expense_matrix, axis=1)
    
    #Total amount each person owes (Column Sum)
    total_owed = np.sum(expense_matrix, axis=0)
    
    # Positive = Should Recieve
    # Negative = Owes Money
    net_balance = total_paid - total_owed
    
    return net_balance

In [143]:
def display_settlements():
    settlements = calculate_settlements()
    
    table = PrettyTable()
    table.field_names = ["Friend", "Settlement"]
    
    for i, friend in enumerate(friends):    # i = index number , friend = name
        
        if settlements[i] > 0:
            table.add_row([friend, f"Should Receive ₹{settlements[i]:.2f}"])
            
        elif settlements[i] < 0:
            table.add_row([friend, f"Owes ₹{-settlements[i]:.2f}"])
            
        else:
            table.add_row([friend, "Is Settled"])
            
    print("\nFinal Setllements:")
    print(table)

In [154]:
# Madhan paid ₹3500 for Madhan, Gokul and Loki
add_expense("Madhan", ["Madhan", "Gokul", "Loki"], 2750) # taxi

# Gokul paid ₹2000 for Gokul and Loki                     #ticket
add_expense("Gokul", ["Gokul", "Loki"], 2500)

# Loki paid ₹1500 for everyone                             #snacks
add_expense("Loki", ["Madhan", "Gokul", "Loki", "Yusuf"], 2000)

In [146]:
display_settlements()


Final Setllements:
+--------+-------------------------+
| Friend |        Settlement       |
+--------+-------------------------+
| Madhan | Should Receive ₹5333.33 |
| Gokul  |       Owes ₹666.67      |
|  Loki  |      Owes ₹2666.67      |
| Yusuf  |      Owes ₹2000.00      |
+--------+-------------------------+


In [163]:
def suggest_payments():

    settlements = calculate_settlements()

    # People who should receive money
    creditors = [
        (friends[i], amt)
        for i, amt in enumerate(settlements)
        if amt > 0                                # positive value = creditor
    ]

    # People who should pay money
    debtors = [
        (friends[i], -amt)
        for i, amt in enumerate(settlements)    #who, amount
        if amt < 0                               # negative value = debtor
    ]

    transactions = []

    # Match debtors with creditors
    while debtors and creditors:             # not empty  #fail

        debtor, debt_amount = debtors.pop(0)
        creditor, credit_amount = creditors.pop(0)

        payment = min(debt_amount, credit_amount)

        transactions.append((debtor, creditor, payment))

        # Update balances
        debt_amount -= payment
        credit_amount -= payment

        # Put back remaining balances
        if debt_amount > 0:
            debtors.insert(0, (debtor, debt_amount))

        if credit_amount > 0:
            creditors.insert(0, (creditor, credit_amount))

    print("\nSuggested Transactions:")

    if transactions:    #not null
        for debtor, creditor, amount in transactions:
            print(f"{debtor} should pay ₹{amount:.2f} to {creditor}")
    else:
        print("No transactions needed. Everyone is settled.")

In [165]:
suggest_payments()


Suggested Transactions:
Gokul should pay ₹1000.00 to Madhan
Loki should pay ₹4000.00 to Madhan
Yusuf should pay ₹3000.00 to Madhan
